# Clustering Fundamentals

### K-Means and DBSCAN

## Import Libraries

We'll use scikit-learn for generating synthetic datasets and matplotlib for visualization throughout this clustering tutorial.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from sklearn.datasets import make_blobs, make_moons

# K-Means Algorithm Implementation

K-Means is an iterative clustering algorithm that partitions data into K clusters by:
1. **Initialization**: Randomly place K centroids
2. **Assignment**: Assign each point to the nearest centroid  
3. **Update**: Move centroids to the center of their assigned points
4. **Repeat**: Until centroids stop moving (convergence)

In [ ]:
def assign_label(x, centroids):
    """ This function computes the clusters, to be precise the index of the cluster for every data point.
    Parameters:
      centroids: array of K centroids with shape (K, 2), i.e. for K=3: [[x1,y1], [x2,y2], [x3,y3]]
      data: array of N data points with shape (n, 2), i.e. [[x1,y1], ..., [xN,yN]]
    Returns:
      array of N cluster indices, indicating the cluster per data point, i.e. [i1, i2, i3, ..., iN]
      0 <= cluster_index < K, i.e. for K=3 the result might look like [0,1,0,2,1,2,0,0,0,...,1] of size N. 
      This would mean that the first point within data corresponds to cluster0, the second to cluster1, and so on.
    """
    # First, compute the distance of every point to the K centroids using np.linalg.norm
    distances = np.linalg.norm(x[:, np.newaxis] - centroids, axis=2)
    
    # Then, set the indices by finding the minimum distance
    cluster_indices = np.argmin(distances, axis=1)
    
    return cluster_indices

# Use this code to plot your clusters.
# Pay attention to pass correct parameters to plot_clusters as described in the method documentation
def plot_clusters(data, cluster_indices, centroids, new_centroids = False):
    """ Plots a scatter plot for three clusters and the corresponding centroids.
    Parameters:
      data: Array of size (N,2) that contains the data
      cluster_indices: Array of size N that contains indices (0,1,2) denoting which data point corresponds to which cluster
      centroids: Array of size (3,2) containing the three centroids of the three clusters
    """
    plt.figure(figsize=(5,4))
    plt.grid(alpha=0.3)
    plt.scatter(data[:,0], data[:,1], c=[list(mcolors.TABLEAU_COLORS.keys())[ci] for ci in cluster_indices])
    plt.scatter(centroids[:,0],centroids[:,1], marker="*", color="k", s=90, label="centroids")
    plt.legend()
    if new_centroids:
        plt.title("Centroids Update")
    else:
        plt.title("Label Update")
    plt.xlabel("$x_1$")
    plt.ylabel("$x_2$")
    # plt.savefig(f"kmeans_{i:2d}.png", format="png", bbox_inches='tight')
    # plt.show()

### Helper Functions

The `assign_label` function implements the assignment step: for each data point, find the closest centroid and assign the point to that cluster. The `plot_clusters` function visualizes the current clustering state.

In [ ]:
def kmeans(X, K, max_iterations=20, random_state=2):
    np.random.seed(random_state)
    idx = np.random.randint(len(X), size=K)
    centroids = X[idx,:]

    # loop until
    # a) convergence (centroids did not change), or
    # b) max_iterations reached
    for i in range(max_iterations):    
        # assign labels based on closest centroid
        labels = assign_label(X, centroids)

        # Label Update Plot
        # plot_clusters(X, labels, centroids)

        # update centroids by computing the mean of the clusters
        new_centroids = np.array([np.mean(X[labels == k,:], axis=0) for k in range(K)])

        # check if centroids have changed - if not: exit the loop (Hint: use the break statement)
        if np.all(new_centroids == centroids):
            break
            
        centroids = new_centroids   

        # Centroids Update Plot
        # plot_clusters(X, labels, centroids, True)

    plot_clusters(X, labels, centroids, True)
    return centroids

### K-Means Main Algorithm

This function implements the complete K-Means algorithm. It alternates between assigning points to clusters and updating centroids until convergence. The visualization shows each iteration step.

## K-Means: Basic Principle

Let's start with a simple example using synthetic data with 2 clear clusters. We'll generate random data points and apply our K-Means implementation to see how it separates the data.

### Generate Synthetic Data

Using `make_blobs` from scikit-learn to create 300 data points arranged in 2 natural clusters. This simulates a typical clustering scenario.

In [ ]:
np.random.seed(6)
X, y = make_blobs(n_samples=300, n_features=2, centers=2, cluster_std=1)

plt.plot(X[:,0], X[:,1], "o", color="gray")
plt.grid(alpha=0.4)
plt.show()

### Apply K-Means Algorithm

Now we run our custom K-Means implementation with K=2 clusters. Watch how the algorithm iteratively improves the cluster assignments and centroid positions.

In [ ]:
centroids = kmeans(X, K=2, random_state=6)

# K-Means: More Clusters

Now let's test K-Means on a more complex dataset with 4 natural clusters. This demonstrates how the algorithm scales to multiple clusters.

### Generate More Complex Data

Creating a dataset with 4 clusters and higher standard deviation, making the clustering task more challenging.

In [ ]:
np.random.seed(8)
X, y = make_blobs(n_samples=300, n_features=2, centers=4, cluster_std=1.6)

### Visualize the Dataset

Before clustering, let's examine the data distribution. Note how the clusters are less separated compared to the previous example.

In [ ]:
# plt.figure(figsize=(8,7))
# plt.scatter(X[:, 0], X[:, 1], c=[list(mcolors.TABLEAU_COLORS.keys())[yi] for yi in y])
# plt.grid(alpha=0.4)
# plt.show()

plt.figure(figsize=(8,7))
plt.grid(alpha=0.3)
plt.xlabel("$x_1$")
plt.ylabel("$x_2$")
plt.plot(X[:, 0], X[:, 1], "o")
plt.show()

### Apply K-Means with K=4

Running K-Means with 4 clusters to match the ground truth. Observe how the algorithm handles overlapping clusters and multiple iterations to convergence.

In [ ]:
centroids = kmeans(X, K=4, random_state=12)

# K-Means with Scikit-Learn

Now let's use the optimized implementation from scikit-learn. This is more efficient and includes advanced features like different initialization methods.

In [ ]:
from sklearn.cluster import KMeans

### Apply Scikit-Learn K-Means

Using `fit_predict()` to both train the model and get cluster assignments in one step. This is the standard way to use K-Means in practice.

In [ ]:
cluster_labels = KMeans(n_clusters=4, random_state=12).fit_predict(X)

In [ ]:
print(cluster_labels)

### Visualize Results

Compare the results with our custom implementation. The cluster labels might be different (0,1,2,3 vs different numbers) but the clustering should be similar.

In [ ]:
plt.figure(figsize=(8,7))
plt.scatter(X[:, 0], X[:, 1], c=cluster_labels)
plt.grid(alpha=0.4)
plt.show()

# Elbow Method

**How do we choose the optimal number of clusters (K)?**

The elbow method plots the sum of squared distances (inertia) for different K values. The "elbow" point suggests the optimal K where adding more clusters doesn't significantly improve the fit.

### Calculate Inertia for Different K Values

We test K from 1 to 10 and plot the inertia (sum of squared distances from points to their assigned centroids). Look for the "elbow" where the curve flattens - this suggests the optimal K.

In [ ]:
N = 10
start = 1
Sum_of_squared_distances = np.zeros(len(range(start,N+1)))

for i, k in enumerate(range(start,N+1)):
    h = KMeans(n_clusters=k, random_state=0)
    h.fit(X)
    Sum_of_squared_distances[i] = h.inertia_


plt.plot(range(start,N+1),Sum_of_squared_distances,"o-")
plt.grid(alpha=.4)
plt.xlabel("K (number of clusters)")
plt.ylabel("sum of squared distances")
plt.xticks(np.arange(1,N+1))
plt.show()

# DBSCAN vs. K-Means

**When K-Means fails:** Not all datasets have spherical clusters! 

**DBSCAN** (Density-Based Spatial Clustering) can find clusters of arbitrary shapes and automatically determines the number of clusters. It's particularly good for:
- Non-spherical clusters
- Clusters of varying densities  
- Datasets with noise/outliers

In [ ]:
from sklearn.cluster import DBSCAN, KMeans

### Load Comparison Datasets

We'll compare two datasets:
- **Moons**: Crescent-shaped clusters that challenge K-Means
- **Blobs**: Traditional spherical clusters where K-Means works well

In [ ]:
x_moons, y_moons = make_moons(n_samples=500, noise=0.1, random_state=42)
x_blobs, y_blobs = make_blobs(n_samples=600, centers=3, cluster_std=1, random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(12,5))
axes[0].scatter(x_moons[:, 0], x_moons[:, 1], c=y_moons, cmap="Set1", s=15)
axes[0].set_title("Moons Data")

axes[1].scatter(x_blobs[:, 0], x_blobs[:, 1], c=y_blobs, s=15, cmap='Set1')
axes[1].set_title("Blobs Data")

for ax in axes:
    ax.set_xticks([])
    ax.set_yticks([])

plt.tight_layout()
plt.show()

### Apply Both Algorithms

**DBSCAN parameters:**
- `eps`: Maximum distance between points in the same cluster
- `min_samples`: Minimum points needed to form a cluster

**K-Means parameters:**
- `n_clusters`: Number of clusters (must be specified)

In [ ]:
# Apply DBSCAN
dbscan_moons = DBSCAN(eps=0.2, min_samples=5).fit_predict(x_moons)
dbscan_blobs = DBSCAN(eps=0.5, min_samples=5).fit_predict(x_blobs)

# Apply k-Means
kmeans_moons = KMeans(n_clusters=2, random_state=0).fit_predict(x_moons)
kmeans_blobs = KMeans(n_clusters=3, random_state=0).fit_predict(x_blobs)

### Compare Results

**Key Observations:**
- **Moons**: DBSCAN correctly identifies the crescent shapes, while K-Means struggles with the non-spherical clusters
- **Blobs**: Both algorithms perform well on spherical clusters
- **Takeaway**: Choose the algorithm based on your data's cluster shape and density patterns

In [ ]:
# Plotting
fig, axes = plt.subplots(2, 2, figsize=(10, 8), sharex=False, sharey=False)

# Titles for columns
axes[0, 0].set_title("Moons")
axes[0, 1].set_title("Blobs")

# Labels for rows
axes[0, 0].set_ylabel("DBSCAN", fontsize=12)
axes[1, 0].set_ylabel("k-Means", fontsize=12)

# Plot DBSCAN results
axes[0, 0].scatter(x_moons[:, 0], x_moons[:, 1], c=dbscan_moons, cmap='Set1', s=15)
axes[0, 1].scatter(x_blobs[:, 0], x_blobs[:, 1], c=dbscan_blobs, cmap='Set1', s=15)

# Plot k-Means results
axes[1, 0].scatter(x_moons[:, 0], x_moons[:, 1], c=kmeans_moons, cmap='Set1', s=15)
axes[1, 1].scatter(x_blobs[:, 0], x_blobs[:, 1], c=kmeans_blobs, cmap='Set1', s=15)

for ax in axes.flat:
    ax.set_xticks([])
    ax.set_yticks([])

plt.tight_layout()
plt.show()